# Databricks PySpark & Delta Lake Practice Guide
**Target Audience:** Azure Data Engineers & Interview Preparation  
**Objective:** Step-by-step hands-on implementation of core lakehouse operations with PySpark & Delta Lake.

### Step-by-Step Roadmap (One Task Per Cell):
1. **Task 1: Environment Setup & Parameterization** (`test.practice.test_table` on ADLS Gen2 `raw` container)
2. **Task 2: Mock DataFrame Generation** (with intentional real-world flaws: NULLs, duplicates, bad casing)
3. **Task 3: Data Hygiene & String Standardization** (`trim`, `initcap`, `lower`, `concat_ws`)
4. **Task 4: Primary Key NULL Validation & Quarantine Isolation** (`df_clean` vs `df_quarantine_nulls`)
5. **Task 5: Duplicate Detection & Window-based Deduplication** (`Window.partitionBy().orderBy()`)
6. **Task 6: Enterprise Audit Metadata Columns** (`_ingested_at`, `_source_file`, `_data_source_mode`, etc.)
7. **Task 7: External Delta Table DDL Creation** (`CREATE TABLE ... USING DELTA LOCATION ...`)
8. **Task 8: Initial Ingestion into External Delta Table** (Writing Batch 1)
9. **Task 9: CDC Batch 2 Generation** (Updates, new Inserts, and new Schema column `CreditScore`)
10. **Task 10: Delta Lake MERGE (Upsert) with Automatic Schema Evolution** (`DeltaTable.merge()`)
11. **Task 11: Delta Lake History & Time Travel Verification** (`DESCRIBE HISTORY`, `VERSION AS OF`)
12. **Task 12: Optional Practice Clean Up / Tear Down**

In [ ]:
# Task 1: Environment Setup & Parameterization
# Configure catalog, schema, table name, and ADLS Gen2 external storage location
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, current_timestamp, trim, initcap, lower, when, row_number, concat_ws, to_timestamp
from pyspark.sql.window import Window
from delta.tables import DeltaTable

# Widgets for interactive parameterization in Databricks
dbutils.widgets.text("target_catalog", "test", "Target Catalog")
dbutils.widgets.text("target_schema", "practice", "Target Schema")
dbutils.widgets.text("target_table", "test_table", "Target Table")
dbutils.widgets.text("storage_account", "azureprojectdevadls01", "Storage Account Name")
dbutils.widgets.text("container", "raw", "Container Name")
dbutils.widgets.text("folder_path", "practice/test_table", "Target Folder Path in Container")

target_catalog  = dbutils.widgets.get("target_catalog").strip()
target_schema   = dbutils.widgets.get("target_schema").strip()
target_table    = dbutils.widgets.get("target_table").strip()
storage_account = dbutils.widgets.get("storage_account").strip()
container       = dbutils.widgets.get("container").strip()
folder_path     = dbutils.widgets.get("folder_path").strip()

full_table_name = f"{target_catalog}.{target_schema}.{target_table}"
external_adls_path = f"abfss://{container}@{storage_account}.dfs.core.windows.net/{folder_path}"
dbfs_fallback_path = f"/tmp/{target_catalog}/{target_schema}/{target_table}"

print("--- Configuration Parameters ---")
print(f"Target Table Full Name : {full_table_name}")
print(f"Target External Path   : {external_adls_path}")
print(f"Fallback DBFS Path     : {dbfs_fallback_path}")

In [ ]:
# Task 2: Create Mock Raw Data with Intentional Real-World Flaws
# Flaws included:
# 1. CustomerID is NULL (tests primary key validation)
# 2. Duplicate CustomerID 101 with different ModifiedDates
# 3. Duplicate CustomerID 102 with identical data
# 4. Whitespaces and erratic casing in text columns
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

raw_data = [
    # CustomerID, FirstName, LastName, Email, City, State, ModifiedDate
    (101, "  John  ", "Doe", "John.Doe@EXAMPLE.com", "  new york  ", "  ny ", "2026-11-20 08:30:00"),
    (101, "John", "Doe", "john.doe@example.com", "New York", "NY", "2026-11-21 14:15:00"),  # Newer update for 101
    (102, "Alice", "Smith", "alice.smith@provider.com", "chicago", "il", "2026-11-20 09:00:00"),
    (None, "Ghost", "User", "ghost@unknown.com", "Austin", "TX", "2026-11-20 10:00:00"),      # NULL Primary Key
    (103, "Bob", "  Johnson ", "BOB.J@COMPANY.ORG", "dallas", "tx", "2026-11-20 11:45:00"),
    (102, "Alice", "Smith", "alice.smith@provider.com", "Chicago", "IL", "2026-11-19 12:00:00"),  # Older duplicate for 102
    (104, "  Carol", "Williams", "carol.w@insurance.net", "san francisco", "ca", "2026-11-21 16:30:00")
]

schema = StructType([
    StructField("CustomerID", IntegerType(), True),
    StructField("FirstName", StringType(), True),
    StructField("LastName", StringType(), True),
    StructField("Email", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("ModifiedDate", StringType(), True)
])

df_raw = spark.createDataFrame(raw_data, schema) \
    .withColumn("ModifiedDate", to_timestamp(col("ModifiedDate"), "yyyy-MM-dd HH:mm:ss"))

print(f"Raw DataFrame created with {df_raw.count()} records:")
df_raw.show(truncate=False)

In [ ]:
# Task 3: Data Cleaning & Standardization
# Apply standard hygiene transformations:
# - trim whitespaces from names, cities, emails, states
# - initcap for Proper Casing (First & Last Names, City)
# - lower for Email address normalization
# - upper for State abbreviation standardization
# - concat_ws for standardized FullName

df_cleaned = df_raw \
    .withColumn("FirstName", initcap(trim(col("FirstName")))) \
    .withColumn("LastName", initcap(trim(col("LastName")))) \
    .withColumn("Email", lower(trim(col("Email")))) \
    .withColumn("City", initcap(trim(col("City")))) \
    .withColumn("State", trim(col("State"))) \
    .withColumn("FullName", concat_ws(" ", col("FirstName"), col("LastName")))

print("--- Cleaned & Standardized DataFrame ---")
df_cleaned.show(truncate=False)

In [ ]:
# Task 4: Primary Key NULL Check & Quarantine Isolation
# Isolate records where CustomerID is NULL into a quarantine DataFrame.
# In production pipelines, quarantined records are routed to an error/bad-data path for alerting.

# 1. Quarantine invalid records
df_quarantine_nulls = df_cleaned.filter(col("CustomerID").isNull()) \
    .withColumn("RejectionReason", lit("Primary Key CustomerID is NULL")) \
    .withColumn("QuarantinedAt", current_timestamp())

# 2. Keep valid records
df_valid_pk = df_cleaned.filter(col("CustomerID").isNotNull())

print(f"Total Incoming Records : {df_cleaned.count()}")
print(f"Valid Records Passed   : {df_valid_pk.count()}")
print(f"Quarantined Records    : {df_quarantine_nulls.count()}")

print("\n--- Quarantined Records Sample ---")
df_quarantine_nulls.show(truncate=False)

In [ ]:
# Task 5: Duplicate Detection & Window-Based Deduplication
# Deduplication Strategy: Partition by Primary Key (CustomerID), order by ModifiedDate DESC,
# and keep the row where row_number() == 1 (the most recent state of the record).

# Step 1: Detect duplicate primary keys
print("--- Records with Duplicate CustomerIDs ---")
df_valid_pk.groupBy("CustomerID").count().filter("count > 1").show()

# Step 2: Apply Window deduplication
window_spec = Window.partitionBy("CustomerID").orderBy(col("ModifiedDate").desc())

df_deduped = df_valid_pk \
    .withColumn("_rn", row_number().over(window_spec)) \
    .filter(col("_rn") == 1) \
    .drop("_rn")

print(f"Deduplicated Record Count: {df_deduped.count()} (Unique Customers)")
print("\n--- Deduplicated DataFrame ---")
df_deduped.show(truncate=False)

In [ ]:
# Task 6: Add Enterprise Audit Metadata Columns
# Add data governance and lineage audit tracking columns

df_with_audit = df_deduped \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", lit("mock_data://practice/customers_batch_01.parquet")) \
    .withColumn("_data_source_mode", lit("Practice_Initial_Load")) \
    .withColumn("InsertedTimestamp", current_timestamp()) \
    .withColumn("UpdatedTimestamp", current_timestamp()) \
    .withColumn("RecordAction", lit("INSERT"))

print("--- Final Batch 1 DataFrame with Audit Columns ---")
df_with_audit.printSchema()
df_with_audit.show(truncate=False)

In [ ]:
# Task 7: Create External Delta Table in Catalog / Schema
# 1. Create Catalog & Schema if permitted by UC permissions
# 2. DDL to create external Delta table at ADLS Gen2 path with fallback support

table_location = external_adls_path

# Create Catalog and Schema if permitted
try:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {target_catalog}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")
    print(f"Catalog & Schema verified: {target_catalog}.{target_schema}")
except Exception as e_cat:
    print(f"[Notice] Unity Catalog creation note: {e_cat}")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_schema}")
    full_table_name = f"{target_schema}.{target_table}"
    print(f"Using fallback schema/table: {full_table_name}")

# DDL for External Delta Table
ddl_sql = f"""
CREATE TABLE IF NOT EXISTS {full_table_name} (
    CustomerID INT,
    FirstName STRING,
    LastName STRING,
    FullName STRING,
    Email STRING,
    City STRING,
    State STRING,
    ModifiedDate TIMESTAMP,
    _ingested_at TIMESTAMP,
    _source_file STRING,
    _data_source_mode STRING,
    InsertedTimestamp TIMESTAMP,
    UpdatedTimestamp TIMESTAMP,
    RecordAction STRING
)
USING DELTA
LOCATION '{table_location}'
"""

try:
    spark.sql(ddl_sql)
    print(f"Table {full_table_name} verified at: {table_location}")
except Exception as e_ddl:
    print(f"[Notice on External Location]: {e_ddl}")
    table_location = dbfs_fallback_path
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {full_table_name} (
        CustomerID INT,
        FirstName STRING,
        LastName STRING,
        FullName STRING,
        Email STRING,
        City STRING,
        State STRING,
        ModifiedDate TIMESTAMP,
        _ingested_at TIMESTAMP,
        _source_file STRING,
        _data_source_mode STRING,
        InsertedTimestamp TIMESTAMP,
        UpdatedTimestamp TIMESTAMP,
        RecordAction STRING
    )
    USING DELTA
    LOCATION '{table_location}'
    """
    )
    print(f"Table created at fallback location: {table_location}")

spark.sql(f"DESCRIBE DETAIL {full_table_name}").select("format", "name", "location").show(truncate=False)

In [ ]:
# Task 8: Initial Ingestion into External Delta Table
# Write Batch 1 to Delta Table using DataFrame writer API

print(f"Writing initial batch to {full_table_name} ({table_location})...")

df_with_audit.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .save(table_location)

print("Initial batch ingestion complete!")

# Verify stored table
df_stored = spark.read.table(full_table_name)
print(f"Total rows in table: {df_stored.count()}")
df_stored.select("CustomerID", "FullName", "Email", "City", "State", "ModifiedDate", "RecordAction").show(truncate=False)

In [ ]:
# Task 9: Generate CDC Batch 2 (Updates, Inserts & Schema Evolution Column)
# Simulate incoming incremental CDC data:
# 1. Update: Customer 101 moves from New York to Boston
# 2. Update: Customer 103 changes Email
# 3. Insert: New Customer 105 (David Brown)
# 4. Schema Evolution: A NEW column `CreditScore` not present in Batch 1

batch2_raw = [
    (101, "John", "Doe", "john.doe@example.com", "Boston", "MA", "2026-11-22 10:00:00", 780),           # UPDATE
    (103, "Bob", "Johnson", "bob.johnson.new@company.org", "Dallas", "TX", "2026-11-22 11:00:00", 710),  # UPDATE
    (105, "David", "Brown", "david.b@cloudtech.io", "Seattle", "WA", "2026-11-22 12:30:00", 820)         # INSERT
]

batch2_schema = StructType([
    StructField("CustomerID", IntegerType(), True),
    StructField("FirstName", StringType(), True),
    StructField("LastName", StringType(), True),
    StructField("Email", StringType(), True),
    StructField("City", StringType(), True),
    StructField("State", StringType(), True),
    StructField("ModifiedDate", StringType(), True),
    StructField("CreditScore", IntegerType(), True)  # <-- NEW COLUMN FOR SCHEMA EVOLUTION
])

df_batch2 = spark.createDataFrame(batch2_raw, batch2_schema) \
    .withColumn("ModifiedDate", to_timestamp(col("ModifiedDate"), "yyyy-MM-dd HH:mm:ss")) \
    .withColumn("FullName", concat_ws(" ", col("FirstName"), col("LastName"))) \
    .withColumn("_ingested_at", current_timestamp()) \
    .withColumn("_source_file", lit("mock_data://practice/customers_batch_02.parquet")) \
    .withColumn("_data_source_mode", lit("Practice_Incremental_CDC")) \
    .withColumn("InsertedTimestamp", current_timestamp()) \
    .withColumn("UpdatedTimestamp", current_timestamp()) \
    .withColumn("RecordAction", lit("PENDING_MERGE"))

print("--- Incoming CDC Batch 2 (With New Column CreditScore) ---")
df_batch2.show(truncate=False)

In [ ]:
# Task 10: Delta Lake MERGE (Upsert) with Automatic Schema Evolution
# Enable autoMerge so incoming new columns are automatically added to the target table
spark.conf.set("spark.databricks.delta.schema.autoMerge.enabled", "true")

target_delta = DeltaTable.forPath(spark, table_location) if "dbfs:" in table_location or "/" in table_location else DeltaTable.forName(spark, full_table_name)

# Perform Upsert (MERGE)
target_delta.alias("tgt") \
    .merge(
        source=df_batch2.alias("src"),
        condition="tgt.CustomerID = src.CustomerID"
    ) \
    .whenMatchedUpdate(set={
        "FirstName": "src.FirstName",
        "LastName": "src.LastName",
        "FullName": "src.FullName",
        "Email": "src.Email",
        "City": "src.City",
        "State": "src.State",
        "ModifiedDate": "src.ModifiedDate",
        "CreditScore": "src.CreditScore",
        "_ingested_at": "src._ingested_at",
        "_source_file": "src._source_file",
        "_data_source_mode": "src._data_source_mode",
        "UpdatedTimestamp": "src.UpdatedTimestamp",
        "RecordAction": lit("UPDATE")
    }) \
    .whenNotMatchedInsert(values={
        "CustomerID": "src.CustomerID",
        "FirstName": "src.FirstName",
        "LastName": "src.LastName",
        "FullName": "src.FullName",
        "Email": "src.Email",
        "City": "src.City",
        "State": "src.State",
        "ModifiedDate": "src.ModifiedDate",
        "CreditScore": "src.CreditScore",
        "_ingested_at": "src._ingested_at",
        "_source_file": "src._source_file",
        "_data_source_mode": "src._data_source_mode",
        "InsertedTimestamp": "src.InsertedTimestamp",
        "UpdatedTimestamp": "src.UpdatedTimestamp",
        "RecordAction": lit("INSERT")
    }) \
    .execute()

print("MERGE completed successfully with Schema Evolution!")

print("\n--- Target Table After MERGE ---")
spark.read.table(full_table_name).orderBy("CustomerID").show(truncate=False)

In [ ]:
# Task 11: Delta Lake Transaction History & Time Travel Verification
# Inspect commit log history and query earlier versions of the table

print("--- Delta Table Commit History ---")
df_history = target_delta.history()
df_history.select("version", "timestamp", "userId", "operation", "operationMetrics").show(truncate=False)

# 1. Query Version 0 (Before MERGE)
print("\n--- Time Travel: Snapshot VERSION AS OF 0 (Before MERGE) ---")
df_v0 = spark.read.format("delta").option("versionAsOf", 0).load(table_location)
df_v0.select("CustomerID", "FullName", "City", "State", "RecordAction").orderBy("CustomerID").show()

# 2. Query Version 1 (After MERGE)
print("\n--- Time Travel: Snapshot Latest Version (After MERGE with CreditScore) ---")
df_v1 = spark.read.format("delta").load(table_location)
df_v1.select("CustomerID", "FullName", "City", "State", "CreditScore", "RecordAction").orderBy("CustomerID").show()

print("Verification successful: Notice John Doe in NY (v0) -> Boston (v1), Customer 105 added, and CreditScore evolved!")

In [ ]:
# Task 12: Optional Practice Cleanup
# Set cleanup_enabled = True when you wish to drop the table and delete storage files

cleanup_enabled = False

if cleanup_enabled:
    print(f"Dropping table {full_table_name}...")
    spark.sql(f"DROP TABLE IF EXISTS {full_table_name}")
    try:
        dbutils.fs.rm(table_location, recurse=True)
        print(f"Storage path cleaned: {table_location}")
    except Exception as e_rm:
        print(f"Storage clean note: {e_rm}")
    print("Practice cleanup completed.")
else:
    print("Cleanup is disabled. Your table, history, and practice files remain ready for exploration!")